In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from lifelines.utils import concordance_index
from sksurv.metrics import integrated_brier_score
from models.survite import SurvITE
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_npz_data(filepath):
    """
    Loads feature matrices and target variables from a NumPy .npz archive.
    """
    # Load the compressed numpy arrays
    data = np.load(filepath)
    
    # Optional: Print the keys to verify what is inside the .npz file
    # print(f"Keys found in {filepath}: {list(data.keys())}")
    
    # Extract the arrays. 
    # NOTE: You may need to change 'x', 't', 'e', 'a' to match the exact 
    # string keys used when the .npz file was created.
    X = data['x'] # Features
    T = data['t'] # Time
    E = data['y'] # Event indicator
    A = data['a'] # Treatment indicator
    
    return X, T, E, A

# Define the paths based on the new folder structure
train_path = os.path.join('data', 'sample', 'tr_data.npz')
test_path = os.path.join('data', 'sample', 'te_data.npz')

# Load train and test sets
X_train, T_train, E_train, A_train = load_npz_data(train_path)
X_test, T_test, E_test, A_test = load_npz_data(test_path)

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

Train features shape: (5000, 10)
Test features shape: (5000, 10)


In [3]:
# Discretize time into T_MAX bins
T_MAX = 30

# Find the global max time to create uniform bins
max_time = max(T_train.max(), T_test.max())
time_bins = np.linspace(0, max_time, T_MAX + 1)

# Function to discretize time for the model
def discretize_time(T, bins):
    # np.digitize returns 1-based indices, we subtract 1 for 0-based
    T_discrete = np.digitize(T, bins) - 1
    # Clip to max index (T_MAX - 1)
    T_discrete = np.clip(T_discrete, 0, T_MAX - 1)
    return T_discrete

T_train_discrete = discretize_time(T_train, time_bins)
T_test_discrete = discretize_time(T_test, time_bins)

# Format time arrays for the model: Add dimension (B, 1)
def format_tensor(X, T, E, A):
    T_expanded = np.expand_dims(T, axis=-1)
    E_expanded = np.expand_dims(E, axis=-1)
    A_expanded = np.expand_dims(A, axis=-1)
    return X, T_expanded, E_expanded, A_expanded

X_train_t, T_train_t, E_train_t, A_train_t = format_tensor(X_train, T_train_discrete, E_train, A_train)
X_test_t, T_test_t, E_test_t, A_test_t = format_tensor(X_test, T_test_discrete, E_test, A_test)

# Format data for sksurv (requires structured arrays)
def get_structured_survival_data(E, T):
    return np.array([(bool(e), t) for e, t in zip(E, T)], 
                    dtype=[('event', '?'), ('time', '<f8')])

train_survival_sksurv = get_structured_survival_data(E_train, T_train)
test_survival_sksurv = get_structured_survival_data(E_test, T_test)

# Evaluation time points for IBS (using the centers of the bins up to the max observed test time)
# sksurv requires eval times to be strictly within the range of train set times
min_train_time = T_train[E_train == 1].min() + 1e-5
max_train_time = T_train.max() - 1e-5
eval_times = time_bins[1:] # Drop the 0 bin
eval_times = eval_times[(eval_times > min_train_time) & (eval_times < max_train_time)]

In [4]:
X_train_t

array([[-0.86089986, -2.07272029, -0.30700235, ..., -0.76261147,
        -0.37664848,  0.46294596],
       [-0.89201575, -1.88150989, -0.92411839, ..., -1.58928698,
         0.06541155,  0.09249244],
       [ 1.22571904,  0.87595506, -0.96365153, ..., -0.206866  ,
         0.69947509, -0.53299544],
       ...,
       [ 1.61543496,  1.46029479,  0.94726507, ..., -0.37694224,
         0.92250726,  0.0087917 ],
       [-0.386835  ,  1.28222598, -1.23494011, ..., -1.39306658,
        -1.38504057, -1.42118923],
       [ 0.83823938,  0.32325428,  3.28606781, ...,  0.76809525,
         0.51030666,  0.79671443]], shape=(5000, 10))

In [5]:
input_dims = {
    'x_dim': X_train.shape[1],
    't_max': T_MAX,
    'num_Event': 1
}

network_settings = {
    'z_dim': 50,
    'h_dim1': 100,
    'h_dim2': 100,
    'num_layers1': 2,
    'num_layers2': 2,
    'active_fn': 'relu',
    'reg_scale': 1e-4,
    'ipm_term': 'wasserstein',
    'is_treat': True,
    'is_smoothing': True
}

model = SurvITE(input_dims, network_settings, device=device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [6]:
EPOCHS = 100
EVAL_INTERVAL = 10
BATCH_SIZE = 256

num_samples = X_train_t.shape[0]

print("Starting Training...")

# Wrap the epoch loop with tqdm
epoch_pbar = tqdm(range(1, EPOCHS + 1), desc="Training Epochs")

for epoch in epoch_pbar:
    model.train()
    
    # Shuffle for batching
    indices = np.random.permutation(num_samples)
    
    total_loss = 0.0
    for start_idx in range(0, num_samples, BATCH_SIZE):
        batch_idx = indices[start_idx:start_idx + BATCH_SIZE]
        
        batch_x = X_train_t[batch_idx]
        batch_y = E_train_t[batch_idx]
        batch_t = T_train_t[batch_idx]
        batch_a = A_train_t[batch_idx]
        
        loss, loss_fact, loss_ipm = model.train_baseline(
            optimizer=optimizer,
            x=batch_x,
            y=batch_y,
            t=batch_t,
            a=batch_a,
            dropout_rate=0.2
        )
        total_loss += loss

    avg_loss = total_loss / num_samples
    
    # Update the progress bar text with the current loss
    epoch_pbar.set_postfix({'Loss': f"{avg_loss:.4f}"})

    # Evaluation
    if epoch % EVAL_INTERVAL == 0 or epoch == EPOCHS:
        model.eval()
        
        # Predict survival for treated and control
        surv_A1 = model.predict_survival_A1(X_test)
        surv_A0 = model.predict_survival_A0(X_test)
        
        # Factual survival (based on actual treatment assignments)
        surv_factual = A_test_t * surv_A1 + (1 - A_test_t) * surv_A0
        
        # 1. Calculate Harrell's C-index
        expected_survival_times = np.sum(surv_factual, axis=1)
        c_index = concordance_index(T_test, expected_survival_times, E_test)
        
        # 2. Calculate Integrated Brier Score (IBS)
        eval_indices = discretize_time(eval_times, time_bins)
        surv_estimates_for_ibs = surv_factual[:, eval_indices]
        
        try:
            ibs_score = integrated_brier_score(
                train_survival_sksurv, 
                test_survival_sksurv, 
                surv_estimates_for_ibs, 
                eval_times
            )
        except Exception as e:
            ibs_score = float('nan')
            tqdm.write(f"  [Warning] IBS calculation failed: {e}")

        # Use tqdm.write instead of print to avoid messing up the progress bar
        tqdm.write(f"Epoch {epoch:03d} | Loss: {avg_loss:.4f} | C-index: {c_index:.4f} | IBS: {ibs_score:.4f}")

Starting Training...


Training Epochs:  10%|█         | 10/100 [00:31<04:42,  3.14s/it, Loss=0.0224]

Epoch 010 | Loss: 0.0224 | C-index: 0.7262 | IBS: 0.1348


Training Epochs:  20%|██        | 20/100 [01:07<04:53,  3.67s/it, Loss=0.0210]

Epoch 020 | Loss: 0.0210 | C-index: 0.7401 | IBS: 0.1265


...

Epoch 080 | Loss: 0.0172 | C-index: 0.7404 | IBS: 0.1263


Training Epochs:  90%|█████████ | 90/100 [05:00<00:33,  3.34s/it, Loss=0.0165]

Epoch 090 | Loss: 0.0165 | C-index: 0.7427 | IBS: 0.1262


Training Epochs: 100%|██████████| 100/100 [05:34<00:00,  3.34s/it, Loss=0.0160]

Epoch 100 | Loss: 0.0160 | C-index: 0.7415 | IBS: 0.1269
